# Sequence-to-Sequence Models with Attention for Machine Translation

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand sequence-to-sequence (seq2seq) encoder–decoder architectures
- **Build a real Keras seq2seq model with attention**, layer by layer, and inspect its structure
- Run a forward pass through the (untrained) model and verify the tensor shapes
- Understand what attention adds to seq2seq, and what actually *training* this model would require

## 🔗 Prerequisites

- ✅ `01_attention_transformers_bridge.ipynb` — attention weights, computed by hand
- ✅ `02_rnn_lstm_nlp.ipynb` — RNN/LSTM hidden states (the building blocks of encoder and decoder)
- ✅ Keras at a black-box level, from Course 01 (AIAT 111), Unit 4

**Kernel note:** this notebook needs TensorFlow — run it on the **tfenv** kernel.

⚠️ **Scope note:** we build and inspect the model but do **not** train it — translation training needs a
large parallel corpus (millions of aligned sentence pairs) and significant compute. Training mechanics
arrive in **AIAT 122 (Course 08, Deep Learning)**.

---

This notebook covers practical activities from **Course 07, Unit 4**:
- Implementing a machine translation model architecture using seq2seq with attention

---

## Introduction

**Sequence-to-Sequence (seq2seq)** models use encoder-decoder architectures to map sequences to sequences. **Attention mechanisms** help models focus on relevant parts of the input when generating output.

## 📥 Inputs & 📤 Outputs

**Inputs:** no data files — a toy configuration (vocabulary 1,000; sequence length 12) and two batches of random token ids, all defined in the code. Requires TensorFlow, so run on the **tfenv** kernel.

**Outputs:** a real `model.summary()` of the seq2seq-with-attention model you build, and an end-to-end forward pass whose output shape and probability rows the code verifies. The weights are untrained (random), so the probabilities are meaningless by design — the architecture is the lesson.

---

In [1]:
# Setup: NumPy plus TensorFlow/Keras (guarded, so the notebook can explain itself
# on machines without TF). Keras supplies the LSTM, Attention, and Dense layers
# we assemble into an encoder-decoder translation model below.

import numpy as np

# Try importing TensorFlow/Keras
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import LSTM, Dense, Embedding, Input, Attention, Concatenate
    HAS_TF = True
    print("✅ TensorFlow/Keras available!")
except ImportError:
    HAS_TF = False
    print("⚠️  TensorFlow not available. Install with: pip install tensorflow")

print("✅ Libraries imported!")

✅ TensorFlow/Keras available!
✅ Libraries imported!


## Part 1: Understanding Seq2Seq Architecture


In [2]:
# Roadmap cell: print the architecture we are about to build, piece by piece.
# Keep this mental picture — encoder reads, attention aligns, decoder writes —
# while you read the real Keras code in the next cell.

print("=" * 60)
print("Sequence-to-Sequence (Seq2Seq) Architecture")
print("=" * 60)

print("\nKey Components:")
print("  1. Encoder: Processes input sequence")
print("  2. Decoder: Generates output sequence")
print("  3. Attention: Connects encoder and decoder")
print("  4. Context Vector: Summary of input sequence")

print("\n" + "-" * 60)
print("Seq2Seq with Attention:")
print("-" * 60)
print("  Encoder: Input sequence → Hidden states")
print("  Attention: Hidden states → Attention weights")
print("  Decoder: Attention weights + Previous output → Next token")
print("  Output: Generated sequence")

print("\n✅ Attention allows model to focus on relevant input parts!")

Sequence-to-Sequence (Seq2Seq) Architecture

Key Components:
  1. Encoder: Processes input sequence
  2. Decoder: Generates output sequence
  3. Attention: Connects encoder and decoder
  4. Context Vector: Summary of input sequence

------------------------------------------------------------
Seq2Seq with Attention:
------------------------------------------------------------
  Encoder: Input sequence → Hidden states
  Attention: Hidden states → Attention weights
  Decoder: Attention weights + Previous output → Next token
  Output: Generated sequence

✅ Attention allows model to focus on relevant input parts!


## Part 2: Building the Seq2Seq-with-Attention Model in Keras

Now we construct the model **for real**. Every layer below is actually created and wired together —
run the cell and read the `model.summary()` table to see the architecture you just built. The weights are
randomly initialized (untrained), so the forward pass at the end produces meaningless probabilities — the
point is that the machine is real, the shapes are right, and you can see exactly where attention sits.


In [3]:
# Build the full encoder-decoder-attention model in Keras and run a forward pass.
# Every component from the roadmap becomes a named layer here; the forward pass
# on dummy data proves the wiring is correct end to end.

if HAS_TF:
    print("=" * 60)
    print("Building Seq2Seq with Attention (for real)")
    print("=" * 60)

    # Toy configuration (a real translation system would use a 30k+ vocabulary)
    vocab_size, embedding_dim, hidden_units, max_len = 1000, 64, 128, 12

    # --- Encoder: reads the source sentence -> sequence of hidden states ---
    encoder_inputs = Input(shape=(max_len,), name="source_tokens")
    encoder_embedding = Embedding(vocab_size, embedding_dim, name="source_embedding")(encoder_inputs)
    encoder_outputs, state_h, state_c = LSTM(
        hidden_units, return_sequences=True, return_state=True, name="encoder_lstm"
    )(encoder_embedding)

    # --- Decoder: generates the target sentence, starting from the encoder's state ---
    decoder_inputs = Input(shape=(max_len,), name="target_tokens")
    decoder_embedding = Embedding(vocab_size, embedding_dim, name="target_embedding")(decoder_inputs)
    decoder_outputs, _, _ = LSTM(
        hidden_units, return_sequences=True, return_state=True, name="decoder_lstm"
    )(decoder_embedding, initial_state=[state_h, state_c])

    # --- Attention: each decoder step looks back over ALL encoder states ---
    context_vector = Attention(name="attention")([decoder_outputs, encoder_outputs])

    # --- Output: decoder state + attention context -> next-token probabilities ---
    decoder_concat = Concatenate(name="concat")([decoder_outputs, context_vector])
    output = Dense(vocab_size, activation="softmax", name="next_token_probs")(decoder_concat)

    model = Model([encoder_inputs, decoder_inputs], output)
    model.compile(optimizer="adam", loss="categorical_crossentropy")

    model.summary()

    # --- Forward pass on dummy data: the machine runs end to end ---
    rng = np.random.default_rng(0)
    src_batch = rng.integers(0, vocab_size, (2, max_len))   # 2 fake source sentences
    tgt_batch = rng.integers(0, vocab_size, (2, max_len))   # 2 fake target prefixes
    predictions = model.predict([src_batch, tgt_batch], verbose=0)

    print(f"\nForward pass: input batch {src_batch.shape} -> output {predictions.shape}")
    print(f"  = (batch of 2, {max_len} decoder steps, probability over {vocab_size} target tokens)")
    print(f"Each step's probabilities sum to 1: {np.allclose(predictions.sum(axis=-1), 1.0)}")
    print(f"Trainable parameters: {model.count_params():,}")

    print("\n⚠️  The weights are random — these probabilities are meaningless until the")
    print("model is trained on a parallel corpus (source/target sentence pairs).")
    print("Training this architecture is AIAT 122 (Course 08) material.")
else:
    print("=" * 60)
    print("Seq2Seq with Attention (TensorFlow required)")
    print("=" * 60)
    print("""
    This notebook needs TensorFlow. Select the tfenv kernel, or install TF:

    1. pip install tensorflow

    2. Re-run this notebook from the top: it will then build the full
       encoder-decoder model with attention and run a forward pass.
    """)

Building Seq2Seq with Attention (for real)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ source_tokens       │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ target_tokens       │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_embedding    │ (None, 12, 64)    │     64,000 │ source_tokens[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ target_embedding    │ (None, 12, 64)    │     64,000 │ target_tokens[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 12, 128), │     98,816 │ source_embedding… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 12, 128), │     98,816 │ target_embedding… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 12, 128)   │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat              │ (None, 12, 256)   │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ next_token_probs    │ (None, 12, 1000)  │    257,000 │ concat[0][0]      │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 582,632 (2.22 MB)

 Trainable params: 582,632 (2.22 MB)

 Non-trainable params: 0 (0.00 B)


Forward pass: input batch (2, 12) -> output (2, 12, 1000)
  = (batch of 2, 12 decoder steps, probability over 1000 target tokens)
Each step's probabilities sum to 1: True
Trainable parameters: 582,632

⚠️  The weights are random — these probabilities are meaningless until the
model is trained on a parallel corpus (source/target sentence pairs).
Training this architecture is AIAT 122 (Course 08) material.


## Summary

### What you actually did here
- Built a **real** Keras seq2seq model: embedding → encoder LSTM → decoder LSTM → attention → softmax output
- Read its `model.summary()` — note where the `attention` layer sits, connecting decoder steps to **all** encoder states
- Ran an end-to-end forward pass and verified the output shape `(batch, steps, vocabulary)`

### Key Concepts
1. **Seq2Seq architecture**: encoder compresses the source; decoder generates the target
2. **Attention**: at every decoding step, the decoder looks back over all encoder hidden states and takes a
   weighted average (the same mechanism you computed by hand in `01_attention_transformers_bridge.ipynb`) —
   this removes the single-context-vector bottleneck and dramatically improves long-sentence translation
3. **What training would need** (not done here): a parallel corpus of aligned sentence pairs, teacher
   forcing, and substantial compute — training mechanics arrive in **AIAT 122 (Course 08)**

### Where this architecture went
Replace the LSTMs with self-attention layers and this encoder–decoder-with-attention design becomes the
**Transformer** ("Attention Is All You Need", 2017) — the architecture behind BERT (previous notebook) and
GPT (next notebook).

**Reference:** Course 07, Unit 4: "Deep Learning for NLP" — seq2seq with attention practical content


## 📚 References

The machine-translation lineage this notebook builds on:

1. Sutskever, I., Vinyals, O., & Le, Q. V. (2014). *Sequence to Sequence Learning with Neural Networks*. NeurIPS 2014. [arXiv:1409.3215](https://arxiv.org/abs/1409.3215)
2. Cho, K., van Merrienboer, B., Gulcehre, C., et al. (2014). *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. EMNLP 2014. [arXiv:1406.1078](https://arxiv.org/abs/1406.1078)
3. Bahdanau, D., Cho, K., & Bengio, Y. (2015). *Neural Machine Translation by Jointly Learning to Align and Translate*. ICLR 2015. [arXiv:1409.0473](https://arxiv.org/abs/1409.0473)
4. NLLB Team et al. (2022). *No Language Left Behind: Scaling Human-Centered Machine Translation*. [arXiv:2207.04672](https://arxiv.org/abs/2207.04672)
